In [ ]:
# from tqdm import tqdm
# from datetime import datetime, timezone
import pandas as pd
import os, warnings
from pandas.errors import ParserWarning
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
warnings.simplefilter('ignore', ParserWarning)
warnings.simplefilter('ignore', UserWarning)
import plotly.io as pio
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import random
from scipy import stats

from PersonAI.anchor import data_dir

# Get Data


In [ ]:
DATA_DIR = data_dir()
data_path = os.path.join(DATA_DIR,"all_aircontrol_with_surveys_hr_FIXED.csv")#os.path.join("Path","to","your","parquet","folder")
df = pd.read_csv(data_path)
print(df.size)
df.head()

# Feature Distribution

In [ ]:
# sns.set()

fig, ax = plt.subplots(2,2,figsize=(10,4))

sns.kdeplot(ax=ax[0,0], data=df,
            x="AirTemp", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[0,1], data=df,
            x="Humidity", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,0], data=df,
            x="CO2", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,1], data=df,
            x="bpm", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)

In [ ]:
rand_user = random.sample(sorted(df.user_id.unique()), k=9)

for feature in ["AirTemp","Humidity","CO2","bpm"]:
        fig, axes = plt.subplots(3, 3, figsize=(18, 10),constrained_layout=True)

        k = 0
        for i in range(3):
                for j in range(3):
                        c = rand_user[k+j]
                        sns.kdeplot(ax=axes[i, j],data = df.loc[df.user_id == c], x=feature, hue="q_preference_temperature" ,
                bw_adjust=0.3,
                linewidth=2.5, fill=True).set_title(c)
                        
                k = k+3


# Linear seperability looking at mean vector


In [ ]:
df_no_na = df.loc[~pd.isna(df.AirTemp) & ~pd.isna(df.Humidity) & ~pd.isna(df.CO2) & ~pd.isna(df.bpm),:]
df_no_na['AirTemp_normalised'], lambda_opt = stats.boxcox(df_no_na.AirTemp)
df_no_na['Humidity_normalised'], lambda_opt = stats.boxcox(df_no_na.Humidity)
df_no_na['CO2_normalised'], lambda_opt = stats.boxcox(df_no_na.CO2)
df_no_na['bpm_normalised'], lambda_opt = stats.boxcox(df_no_na.bpm)

In [ ]:
df_no_na.head()
# sns.kdeplot(data=df_no_na,
#             x="AirTemp", hue="q_preference_temperature" ,
#             bw_adjust=0.3,
#             linewidth=2.5, fill=True)

In [ ]:
# sns.set()

fig, ax = plt.subplots(2,2,figsize=(10,4),constrained_layout=True)

sns.kdeplot(ax=ax[0,0], data=df_no_na,
            x="AirTemp_normalised", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[0,1], data=df_no_na,
            x="Humidity_normalised", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,0], data=df_no_na,
            x="CO2_normalised", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)
sns.kdeplot(ax=ax[1,1], data=df_no_na,
            x="bpm_normalised", hue="q_preference_temperature" ,
            bw_adjust=0.3,
            linewidth=2.5, fill=True)

In [ ]:
df_no_na.head()


In [ ]:
mean_vector = df_no_na[['q_preference_temperature','AirTemp_normalised','Humidity_normalised','CO2_normalised','bpm_normalised']].groupby("q_preference_temperature").mean()#.agg(["mean"])
mean_vector

In [ ]:
mean_vector = mean_vector.reset_index(drop = False)
mean_vector.columns = ['q_preference_temperature','AirTemp_mean','Humidity_mean','CO2_mean','bpm_mean']

mean_vector_stack = mean_vector.set_index(['q_preference_temperature']).stack().reset_index()

mean_vector_stack.columns = ["preference_temperature","feature","mean_normalised"]

sns.barplot(mean_vector_stack, x="feature", y="mean_normalised", hue = "preference_temperature")

In [ ]:
df_no_na = df_no_na.merge(mean_vector, on = ['q_preference_temperature'])
df_no_na['AirTemp_diff'] = df_no_na['AirTemp_normalised'] - df_no_na['AirTemp_mean']
df_no_na['Humidity_diff'] = df_no_na['Humidity_normalised'] - df_no_na['Humidity_mean']
df_no_na['CO2_diff'] = df_no_na['CO2_normalised'] - df_no_na['CO2_mean']
df_no_na['bpm_diff'] = df_no_na['bpm_normalised'] - df_no_na['bpm_mean']

df_no_na_stack = df_no_na[['q_preference_temperature','AirTemp_diff','Humidity_diff','CO2_diff','bpm_diff']].set_index(['q_preference_temperature']).stack().reset_index()

df_no_na_stack.columns = ["preference_temperature","feature","diff_to_mean"]


In [ ]:
sns.stripplot(
    data=df_no_na_stack, x="feature", y="diff_to_mean", hue="preference_temperature",
    dodge=False, alpha=.2, legend=False,
)
# sns.pointplot(
#     data=mean_vector_stack, x="feature", y="mean_normalised", hue="preference_temperature",
#     dodge=.4, linestyle="none", errorbar=None,
#     marker="_", markersize=20, markeredgewidth=3,
# )

In [ ]:
sns.stripplot(
    data=df_no_na_stack.loc[df_no_na_stack.feature != 'Humidity_diff',:], x="feature", y="diff_to_mean", hue="preference_temperature",
    dodge=False, alpha=.2, legend=False,
)

In [ ]:
sns.stripplot(
    data=df_no_na_stack.loc[(df_no_na_stack.feature == 'bpm_diff') | (df_no_na_stack.feature == 'CO2_diff'),:], x="feature", y="diff_to_mean", hue="preference_temperature",
    dodge=False, alpha=.2, legend=False,
)

In [ ]:
sns.stripplot(
    data=df_no_na_stack.loc[(df_no_na_stack.feature == 'CO2_diff'),:], x="feature", y="diff_to_mean", hue="preference_temperature",
    dodge=False, alpha=.2, legend=False,
)

In [ ]:
#Normalize everything together


In [ ]:
df_no_na = df_no_na.reset_index(drop = True).reset_index()
df_no_na = df_no_na.rename(columns = {"index":"survey_index"})
df_no_na


In [ ]:
df_stack = df_no_na[['survey_index','q_preference_temperature','AirTemp','Humidity','CO2','bpm']].set_index(['survey_index','q_preference_temperature']).stack().reset_index()

df_stack.columns = ["survey_index","preference_temperature","feature","value"]

df_stack['value_normalised'], lambda_opt = stats.boxcox(df_stack.value)
df_stack

In [ ]:
# df_stack = df_stack.reset_index(drop = True) #.set_index(['preference_temperature'], append=True).index#.pivot(index = [0],columns='feature', values=['preference_temperature','value_normalised'])
# df_stack.head()
df_pivot = df_stack.pivot(index = ['survey_index','preference_temperature'],columns='feature', values=['value_normalised']).reset_index()
df_pivot.columns = df_pivot.columns.droplevel()
df_pivot.columns = ["survey_index","preference_temperature","AirTemp","CO2","Humidity","bpm"]


In [ ]:
mean_vector = df_pivot[['preference_temperature','AirTemp','Humidity','CO2','bpm']].groupby("preference_temperature").mean().reset_index()#.agg(["mean"])

mean_vector.columns = ['preference_temperature','AirTemp_mean','Humidity_mean','CO2_mean','bpm_mean']


df_pivot = df_pivot.merge(mean_vector, on = ['preference_temperature'])
df_pivot['AirTemp_diff'] = df_pivot['AirTemp'] - df_pivot['AirTemp_mean']
df_pivot['Humidity_diff'] = df_pivot['Humidity'] - df_pivot['Humidity_mean']
df_pivot['CO2_diff'] = df_pivot['CO2'] - df_pivot['CO2_mean']
df_pivot['bpm_diff'] = df_pivot['bpm'] - df_pivot['bpm_mean']

df_pivot_stack = df_pivot[['preference_temperature','AirTemp_diff','Humidity_diff','CO2_diff','bpm_diff']].set_index(['preference_temperature']).stack().reset_index()

df_pivot_stack.columns = ["preference_temperature","feature","diff_to_mean"]

In [ ]:
sns.stripplot(
    data=df_pivot_stack, x="feature", y="diff_to_mean", hue="preference_temperature",
    dodge=False, alpha=.2, legend=False,
)

In [ ]:
## per user


In [ ]:
df_stack = df_no_na[['survey_index','user_id','q_preference_temperature','AirTemp','Humidity','CO2','bpm']].set_index(['survey_index','user_id','q_preference_temperature']).stack().reset_index()

df_stack.columns = ["survey_index","user_id","preference_temperature","feature","value"]

df_stack['value_normalised'], lambda_opt = stats.boxcox(df_stack.value)
df_stack

In [ ]:

df_pivot = df_stack.pivot(index = ['survey_index','user_id','preference_temperature'],columns='feature', values=['value_normalised']).reset_index()
df_pivot.columns = df_pivot.columns.droplevel()
df_pivot.columns = ["survey_index","user_id","preference_temperature","AirTemp","CO2","Humidity","bpm"]

df_pivot.head()

In [ ]:
mean_vector = df_pivot[['preference_temperature','AirTemp','Humidity','CO2','bpm']].groupby("preference_temperature").mean().reset_index()#.agg(["mean"])

mean_vector.columns = ['preference_temperature','AirTemp_mean','Humidity_mean','CO2_mean','bpm_mean']

user_mean_vector = df_pivot[["user_id","preference_temperature","AirTemp","CO2","Humidity","bpm"]].groupby(["user_id","preference_temperature"]).mean().reset_index()#.agg(["mean"])
user_mean_vector.columns = ['user_id','preference_temperature','AirTemp_user_mean','Humidity_user_mean','CO2_user_mean','bpm_user_mean']

df_pivot = df_pivot.merge(mean_vector, on = ['preference_temperature'])

user_mean_vector = user_mean_vector.merge(mean_vector, on = ['preference_temperature'])

user_mean_vector['AirTemp_user_diff'] = user_mean_vector['AirTemp_user_mean'] - user_mean_vector['AirTemp_mean']
user_mean_vector['Humidity_user_diff'] = user_mean_vector['Humidity_user_mean'] - user_mean_vector['Humidity_mean']
user_mean_vector['CO2_user_diff'] = user_mean_vector['CO2_user_mean'] - user_mean_vector['CO2_mean']
user_mean_vector['bpm_user_diff'] = user_mean_vector['bpm_user_mean'] - user_mean_vector['bpm_mean']

user_mean_vector[['user_id','preference_temperature','AirTemp_user_diff','Humidity_user_diff','CO2_user_diff','bpm_user_diff']]


In [ ]:
user_mean_vector_stack = user_mean_vector[['user_id','preference_temperature','AirTemp_user_diff','Humidity_user_diff','CO2_user_diff','bpm_user_diff']].set_index(['user_id','preference_temperature']).stack().reset_index()
user_mean_vector_stack
user_mean_vector_stack.columns = ["user_id","preference_temperature","feature","diff_to_mean"]

In [ ]:
sns.stripplot(
    data=user_mean_vector_stack, x="feature", y="diff_to_mean", hue="preference_temperature",
    dodge=False, alpha=.2, legend=False,
)
plt.tight_layout()